# train4: temporary training notebook (synthetic FineWeb QA data from the advisor, contrastive trained together with the distillation)

Data: the synthetic FineWeb QA file the advisor shared (322,710 Turkish web documents, each with an LLM-generated summary, keywords, sentiment, category, three question/answer pairs and three search queries). Cleaned and decontaminated locally with `turkish_distill.prepare_synth_qa` (keywords dropped; generated texts with Cyrillic/CJK leaks dropped; documents sharing a 13-gram with the TR-MTEB + Mizan test texts or the two RAG sets removed: 1,162, mostly WebFAQ 775 and STS22 149): 319,728 documents, 922,676 query -> document pairs, 919,660 question -> document pairs with answers, 297,620 summary -> document pairs (passages cut to 2,500 characters). On Drive under `bitirme/data/fineweb_qa/`; these runs read 300K-pair random subsets (`sub/`), about twice what 5400 steps use, to keep the RAM low.

Two runs, the plain two-teacher recipe (1k-base student, BGE-M3 + plain 8K-embed, 200K mix, lr 1e-4, 5400 steps, seed 42) with the contrastive term of `joint` (every 4th step a batch of 128 pairs, weight 0.3, InfoNCE scale 20, in-batch negatives):
1. `fw_joint_query_w03_s5400`: search query -> document pairs.
2. `fw_joint_qa_ans_w03_s5400`: question -> document pairs plus the answer term (question -> answer, fixed weight 0.3): the advisor's answer idea again, now with sentence-long answers (median 83 characters; the stage-1 QA answers were 2 words).

References: the plain pair `scale_two_lr1e4_mix200k_s5400`, TR-MTEB 67.16 (seed 43: 67.25), and the same joint setting with the 38K stage-1 QA pairs `joint_w03_e4`, 67.31: the only difference to the latter is the pair data. Only differences above 0.3 are read. The joint steps add ~25% to the run time (~50 min each on an A100). Re-running the cells after a disconnect resumes the unfinished run and skips finished ones, then the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook, one after another. (name, config, extra overrides). A later override wins.
CONFIG = 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml'   # BGE-M3 + plain 8K-embed
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.max_steps=5400", "train.lr=0.0001"]
FW = f'{DRIVE_ROOT}/data/fineweb_qa/sub'
assert os.path.exists(f'{FW}/pairs_query_train_300k.jsonl') and os.path.exists(f'{FW}/pairs_qa_train_300k.jsonl'), FW
JOINT = ['joint.weight=0.3', 'joint.every=4', 'joint.batch_size=128', 'joint.scale=20']
RUNS = [
    ('fw_joint_query_w03_s5400', CONFIG, JOINT + [f'joint.pairs={FW}/pairs_query_train_300k.jsonl']),
    ('fw_joint_qa_ans_w03_s5400', CONFIG, JOINT + [f'joint.pairs={FW}/pairs_qa_train_300k.jsonl',
                                                  'joint.answer_weight_mode=fixed', 'joint.answer_weight=0.3']),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()